# Co-occurrence: which seizure codes occur together

Correlation (phi: Pearson's r on present / absent) between every pair of codes that at least 5 patients have before surgery, plus **LGS** (recorded for all 100) as one more row and column, in black. Near 0: the two occur together about as often as chance would give; positive: together more often; negative: rarely together. Tick labels are coloured by core group, to compare the data's grouping with the core groups.

**The focal awareness codes are merged into the feature codes.** Under the ILAE 2017 taxonomy a focal seizure is described twice: by awareness (focal motor / non-motor, aware / impaired awareness: codes 14, 15, 25, 26) and by its onset feature (tonic, clonic, to BTC, behaviour arrest, ...). Those pairs correlate almost perfectly by construction (e.g. focal non-motor impaired awareness with behaviour arrest, phi 0.97), and they are why a patient has a median of 4 codes. So the awareness codes are left out, and each focal seizure counts once, through its feature. Nobody is lost by this except 2 patients whose focal motor seizures have an awareness code but no motor feature code.

The codes are reordered by **hierarchical clustering** (average linkage on 1 - phi), so codes that occur together sit next to each other; the tree shows where they join. With 100 patients and correlations mostly below 0.3, treat the tree's lower branches as loose.

`code_correlations.py` computes the correlations, and this notebook runs it. The types counterpart is `4_combinations/types/cooccurrence.ipynb`. The output goes to `results/<version>/4_combinations/codes/`.

In [ ]:
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cc_stats.paths import DATASET, PROJECT_ROOT, results_dir
from cc_stats.plots import GROUP_COLORS, INK, save_figure

print(f'dataset {DATASET}')
run = subprocess.run([sys.executable, 'analysis/4_combinations/code_correlations.py'],
                     capture_output=True, text=True, cwd=PROJECT_ROOT)
if run.returncode != 0:
    raise RuntimeError(f'code_correlations.py failed:\n{run.stderr}')
RESULTS = results_dir('4_combinations/codes')
codes = pd.read_csv(results_dir('0_cohort/codes') / 'code_counts.csv', keep_default_na=False)

In [ ]:
pairs = pd.read_csv(RESULTS / 'code_correlations.csv', dtype={'code_a': str, 'code_b': str})
# Codes are numbers, LGS is the one named row
key = lambda v: int(v) if v.isdigit() else v
pairs[['code_a', 'code_b']] = pairs[['code_a', 'code_b']].map(key)
codes_in = sorted({c for c in pd.concat([pairs['code_a'], pairs['code_b']]) if c != 'LGS'})
common = codes_in + ['LGS']
phi = pd.DataFrame(np.nan, index=common, columns=common)
for r in pairs.itertuples():
    phi.loc[r.code_b, r.code_a] = r.phi  # lower triangle only
label = codes.set_index('code')['label'].to_dict() | {'LGS': 'LGS'}
group = codes.set_index('code')['core_group'].to_dict() | {'LGS': ''}
TICK = lambda c: 'black' if c == 'LGS' else GROUP_COLORS.get(group[c], '#555555')

from scipy.cluster.hierarchy import dendrogram, leaves_list, linkage
from scipy.spatial.distance import squareform

AWARENESS = [14, 15, 25, 26]
features = [c for c in common if c not in AWARENESS]
full = pd.DataFrame(1.0, index=features, columns=features)
for r in pairs.itertuples():
    if r.code_a in features and r.code_b in features:
        full.loc[r.code_a, r.code_b] = full.loc[r.code_b, r.code_a] = r.phi

tree = linkage(squareform(1 - full.to_numpy(), checks=False), method='average')
order = [features[i] for i in leaves_list(tree)]
m = full.loc[order, order]

fig = plt.figure(figsize=(13, 12.5))
grid = fig.add_gridspec(2, 2, height_ratios=[1.3, 8], width_ratios=[8, 0.25], hspace=0.02, wspace=0.03)
ax_tree, ax, cax = fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[1, 0]), fig.add_subplot(grid[1, 1])
dendrogram(tree, ax=ax_tree, color_threshold=0, above_threshold_color=INK, no_labels=True)
# dendrogram puts leaf i at x = 10i + 5; match the heatmap's columns
ax_tree.set_xlim(0, 10 * len(order))
ax_tree.set_axis_off()
im = ax.imshow(m.to_numpy(dtype=float), cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
for i in range(len(order)):
    for j in range(len(order)):
        if i != j:
            v = m.iat[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7, color='white' if abs(v) > 0.6 else 'black')
# Patients with each code (or LGS), from the pair counts
n_of = {**dict(zip(pairs['code_a'], pairs['n_a'])), **dict(zip(pairs['code_b'], pairs['n_b']))}
named = [f'{label[c]} (n={n_of[c]})' for c in order]
ax.set_xticks(range(len(order)), named, fontsize=9, rotation=45, ha='right', rotation_mode='anchor')
ax.set_yticks(range(len(order)), named, fontsize=9)
for ticks in (ax.get_xticklabels(), ax.get_yticklabels()):
    for tick, c in zip(ticks, order):
        tick.set_color(TICK(c))
        tick.set_fontweight('bold' if c == 'LGS' else 'normal')
ax.spines[:].set_visible(False)
fig.colorbar(im, cax=cax, label='phi (correlation of present / absent)')
ax_tree.set_title('Seizure codes and LGS clustered by how often they occur together (focal awareness codes merged)', loc='left')
save_figure('4_combinations/codes', 'code_correlations_clustered')
plt.show()